# Task B -- full-data R-Drop submission

This notebook trains the R-Drop version of the current best Task B recipe and writes one
submission ZIP. It is intentionally a final-fit run, not an OOF experiment: all labelled
Task B rows are used for training, so no local macro-F1 is available.

Recipe:
- TAPT MuRIL on all permitted Task B and OffensEval text
- one encoder layer reinitialized
- R-Drop KL weight 0.5
- five classifier seeds: 42, 43, 44, 45, 46
- all 3,159 labelled Task B rows, with duplicates retained to match the submitted fit
- six epochs, mean+max pooling, FGM, EMA, balanced class weighting, no auxiliary head

The five validation probability matrices are averaged, then converted to the required
395-row id,label file. The final validated file is
b_reinit1_rdrop_full.zip. Upload that ZIP to the Task B CodaBench phase after reviewing
the log. Expected runtime is approximately 4--6 hours on a T4.


In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Run full-data TAPT

The TAPT input is the complete permitted corpus: multiclass_train.csv plus
offenseval_kn.csv. Labels are not used by the masked-language-model stage. The
min-words and deduplication flags match the earlier full-data submission, so TAPT uses
all 6,406 allowed comments and does not hold out a perplexity split.


In [ ]:
TAPT_OUT = "artifacts/runs/tapt-d0v0-rdrop-full"
TAPT_LOG = "artifacts/logs/rdrop_full_tapt.log"
if (pathlib.Path(TAPT_OUT) / "config.json").exists():
    print("using existing TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--corpus", "data/raw/multiclass_train.csv",
         "data/external/offenseval_kn.csv",
         "--val-frac", "0", "--min-words", "1", "--no-dedupe",
         "--out", TAPT_OUT], log=TAPT_LOG)
assert (pathlib.Path(TAPT_OUT) / "config.json").exists(), "TAPT checkpoint was not written"
print("full-data TAPT checkpoint ready:", TAPT_OUT)


## 2. Train five full-data R-Drop models

folds=1 means there is no validation split: each seed trains on every labelled row and
contributes validation-input probabilities to the five-seed average. Because this is a
final fit, the notebook does not calculate an OOF score or choose a checkpoint using
hidden labels.


In [ ]:
TAG = "b_reinit1_rdrop_full"
run_dir = pathlib.Path("artifacts/runs") / TAG
run([sys.executable, "-u", "-m", "hastika.task_b.train",
     "--tag", TAG,
     "--model", TAPT_OUT,
     "--folds", "1",
     "--no-dedupe",
     "--reinit-layers", "1",
     "--rdrop", "0.5",
     "--aux-weight", "0",
     "--seeds", "42", "43", "44", "45", "46",
     "--epochs", "6"],
    log=f"artifacts/logs/{TAG}.log")
assert (run_dir / "predictions.csv").exists(), "full-data predictions were not written"
assert (run_dir / "test_probs.npy").exists(), "full-data probabilities were not written"
print("five-seed full-data R-Drop fit completed:", run_dir)


## 3. Validate and package the CodaBench submission

The submission helper checks the 395 validation IDs, allowed six-way labels, and exact
id,label header before writing a ZIP containing one bare predictions.csv.


In [ ]:
PRED = pathlib.Path("artifacts/runs") / TAG / "predictions.csv"
ZIP = pathlib.Path("/kaggle/working") / f"{TAG}.zip"
run([sys.executable, "-m", "hastika.common.submission",
     "--task", "b", "--pred", str(PRED), "--out", str(ZIP)])

with zipfile.ZipFile(ZIP) as z:
    assert z.namelist() == ["predictions.csv"], z.namelist()
print("READY TO UPLOAD:", ZIP)


## 4. Preserve the submission output

Download the output directory from Kaggle. The ZIP is the only file needed for CodaBench;
the log, prediction CSV, probability matrix, and TAPT log are included for reproducibility.


In [ ]:
OUT = pathlib.Path("/kaggle/working/rdrop_full_submission")
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / ZIP.name)
shutil.copy2(PRED, OUT / PRED.name)
shutil.copy2(pathlib.Path("artifacts/runs") / TAG / "test_probs.npy",
             OUT / f"{TAG}_test_probs.npy")
shutil.copy2(pathlib.Path("artifacts/logs") / f"{TAG}.log",
             OUT / f"{TAG}.log")
if pathlib.Path(TAPT_LOG).exists():
    shutil.copy2(TAPT_LOG, OUT / pathlib.Path(TAPT_LOG).name)
print("download:", OUT / ZIP.name)
print("all reproducibility files:", OUT)
